# 🎬 Faceless Review Video Generator — 1-Click Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Motchucuncon/AllInOne/blob/master/run_pipeline.ipynb)

**Tự động tạo video Review bằng AI — chỉ cần 1 lần chạy!**

---
### ⚙️ 1. Thiết lập Runtime
`Runtime` → `Change runtime type` → Chọn **T4 GPU**

### 🚀 2. Chạy cell duy nhất bên dưới
Điền thông tin → Ấn Play → Nhận kết quả
---

In [ ]:
# @title 🚀 1-CLICK: TẠO VIDEO REVIEW (Ấn Play để chạy)

# ============================================================
# CẤU HÌNH ĐẦU VÀO (Điền thông tin của bạn vào đây)
# ============================================================

# @markdown ---
TOPIC = "Đánh giá iPhone 15 Pro Max"  # @param {type:"string"}
# @markdown Chủ đề Review (VD: "Đánh giá iPhone 15 Pro Max", "Top 5 quán cà phê Sài Gòn")

VIDEO_MODEL = "wan_2_1"  # @param ["wan_2_1", "ltx_video"]
# @markdown wan_2_1 = chất lượng cao, ltx_video = nhanh hơn

OPENROUTER_MODEL = "deepseek/deepseek-r1:free"  # @param ["deepseek/deepseek-r1:free", "deepseek/deepseek-chat:free", "meta-llama/llama-3.3-70b-instruct:free", "mistralai/mistral-7b-instruct:free", "google/gemini-2.0-flash-lite-preview:free", "qwen/qwen-2.5-72b-instruct:free"]

TTS_VOICE = "vi-VN-HoaiMyNeural"  # @param ["vi-VN-HoaiMyNeural", "vi-VN-NamMinhNeural"]
# @markdown HoaiMy = giọng nữ, NamMinh = giọng nam

# ============================================================
# XỬ LÝ
# ============================================================

import os, sys, subprocess, json, time, warnings
from IPython.display import display, HTML, clear_output, Video as IPythonVideo

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

print("=" * 60)
print("🎬 FACELESS REVIEW VIDEO GENERATOR")
print("=" * 60)

# --- 1. Install dependencies ---
print("\n[1/5] 📦 Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "gradio>=4.0.0", "requests", "diffusers", "transformers",
    "accelerate", "torch", "torchvision", "edge-tts",
    "moviepy", "ffmpeg-python", "imageio-ffmpeg", "pillow", "numpy",
], capture_output=True)
print("   ✅ Dependencies installed")

# --- 2. Clone repo ---
print("\n[2/5] 📂 Cloning repository...")
REPO_DIR = "/content/AllInOne"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/Motchucuncon/AllInOne.git", REPO_DIR],
                   check=True, capture_output=True)
    print("   ✅ Repository cloned")
else:
    print("   ✅ Repository already exists")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# --- 3. Get API key ---
print("\n[3/5] 🔑 OpenRouter API Key...")
from getpass import getpass
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    API_KEY = getpass("   Nhập OpenRouter API key (https://openrouter.ai/keys): ")
    if not API_KEY:
        print("   ⚠️  Không có API key! Dùng chế độ mock (offline).")
    os.environ["OPENROUTER_API_KEY"] = API_KEY
print("   ✅ OK" if API_KEY else "   ⚠️  Mock mode")

# --- 4. Run pipeline ---
print("\n[4/5] 🎬 Running pipeline...")

from core.script_gen import generate_storyboard
from core.audio_gen import generate_audio_and_subtitles
from core.image_gen import generate_broll_images
from core.video_gen import render_video_clips
from core.composer import compose_final_video

os.makedirs("output", exist_ok=True)

# --- Storyboard ---
print("\n   🤖 STEP 1/5: Storyboard generation...")
storyboard = generate_storyboard(topic=TOPIC, model=OPENROUTER_MODEL, api_key=API_KEY or None)
with open("output/storyboard.json", "w", encoding="utf-8") as f:
    json.dump(storyboard, f, ensure_ascii=False, indent=2)
scenes = storyboard.get("storyboard_scenes", [])
print(f"      ✅ {len(scenes)} scenes generated")
if not scenes:
    raise ValueError("No scenes in storyboard!")

# --- Audio ---
print("   🔊 STEP 2/5: Audio generation...")
audio = generate_audio_and_subtitles(storyboard=storyboard, voice=TTS_VOICE, output_dir="output")
print(f"      ✅ Audio: {audio['duration_seconds']:.1f}s")

# --- Images ---
print("   🖼️  STEP 3/5: B-roll image generation...")
prompts = [s["broll_prompt"] for s in scenes]
ids = [s["scene_id"] for s in scenes]
images = generate_broll_images(prompts=prompts, scene_ids=ids,
    output_dir="output/images", unload_after=True)
print(f"      ✅ {len(images)} images generated")

# --- Video ---
print(f"   🎬 STEP 4/5: Video rendering ({VIDEO_MODEL})...")
clips = render_video_clips(image_paths=images, prompts=prompts, scene_ids=ids,
    model=VIDEO_MODEL, output_dir="output/clips")
print(f"      ✅ {len(clips)} clips rendered")

# --- Compose ---
print("   🎞️  STEP 5/5: Composing final video...")
final = compose_final_video(clip_paths=clips, audio_path=audio["audio_path"],
    subtitles_path=audio["subtitles_path"], output_dir="output")
print(f"      ✅ Final video: {final}")

# --- Done ---
print("\n" + "=" * 60)
print("🎉 PIPELINE COMPLETE!")
print("=" * 60)

# --- Display results ---
clear_output(wait=True)

display(HTML(f"""
<div style='background:linear-gradient(135deg,#1a1a2e,#16213e);padding:25px;border-radius:12px;color:white;font-family:sans-serif;margin:10px 0;'>
  <h2 style='margin:0 0 15px 0;'>🎉 VIDEO REVIEW ĐÃ ĐƯỢC TẠO!</h2>
  <hr style='border-color:#333;margin:10px 0;'>
  <table style='width:100%;color:#ddd;'>
    <tr><td style='padding:4px 8px;'>📝 <b>Chủ đề:</b></td><td>{TOPIC}</td></tr>
    <tr><td style='padding:4px 8px;'>🎬 <b>Model Video:</b></td><td>{VIDEO_MODEL}</td></tr>
    <tr><td style='padding:4px 8px;'>📋 <b>Số scene:</b></td><td>{len(scenes)}</td></tr>
    <tr><td style='padding:4px 8px;'>⏱️ <b>Thời lượng:</b></td><td>{audio['duration_seconds']:.1f} giây</td></tr>
    <tr><td style='padding:4px 8px;'>📁 <b>Video:</b></td><td><code>{final}</code></td></tr>
  </table>
  <hr style='border-color:#333;margin:10px 0;'>
  <p style='color:#aaa;font-size:14px;'>👇 Chạy cell bên dưới để xem video và tải về</p>
</div>
"""))

---
## 📥 Xem và Tải Video

In [ ]:
# @title 📥 Xem video và tải về máy
import os
from google.colab import files
from IPython.display import display, Video as IPythonVideo, HTML

final_path = "/content/AllInOne/output/final_review_video.mp4"

if os.path.exists(final_path):
    size_mb = os.path.getsize(final_path) / (1024 * 1024)
    display(HTML(f"<h3>🎥 Video Review — {size_mb:.1f} MB</h3>"))
    display(IPythonVideo(final_path, width=720))
    print("\n📥 Click nút bên dưới để tải video về máy:")
    files.download(final_path)
else:
    print("❌ Chưa có video. Chạy cell đầu tiên trước.")